# STEP 7 — Robustness: ablation + confidence interval (CI) + significance

Closes the "are the results real or chance?" question. Model: RAW LightGBM (Step 2
best_params). 5 seeds (config.seeds = [42, 7, 123, 2024, 99]); all repeated results are
mean ± std + 95% CI. Sets are separate; no merging. The heavy logic lives in `src/robustness.py`.

- **Part 1 — Profit-chain ablation** (telco + cell2cell): remove/break each component of the thesis,
  measure the effect in MONEY (profit/ROI). Reference: c=5% avg-CLV, γ=0.30, horizon 24 months.
- **Part 2 — Repeated runs + CI** (5 sets): are the main metrics stable against the seed?
- **Part 3 — Significance**: are model/strategy differences real or noise (Wilcoxon).

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


def _bul_kok():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "config.yaml").exists():
            return c
    raise RuntimeError("config.yaml not found")


KOK = _bul_kok()
if str(KOK) not in sys.path:
    sys.path.insert(0, str(KOK))

warnings.filterwarnings("ignore")
from src import config as cfg
from src import robustness as rb
from src import strings as S

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

CIKTI = []


def yaz(s=""):
    print(s)
    CIKTI.append(str(s))


veriler = {k: pd.read_csv(cfg.PROCESSED / f"{k}_clean.csv") for k in cfg.DATASETS}
yaz(f"Seeds: {rb.SEEDS} | reference c=%{int(rb.C_ORAN*100)} avg-CLV, γ={rb.GAMMA}")

Seeds: [42, 7, 123, 2024, 99] | reference c=%5 avg-CLV, γ=0.3


## Part 1 — Profit-chain ablation (telco + cell2cell)
K0 full system (raw + profit-threshold); K1 −profit-threshold (0.5); K2 −calibration (class-weight);
K3 −strong model (LogReg); K4 +imbalance (SMOTE). Each with 5 seeds.

In [2]:
yaz(S.MSG["bolum"].format(ad="PART 1 — PROFIT-CHAIN ABLATION"))
tum_abl = {}
for s in rb.ABLATION_SETLERI:
    t = time.time()
    tum_abl[s] = rb.ablation_set(s, veriler[s])
    for kosul, v in tum_abl[s].items():
        yaz(S.MSG7["ablation"].format(set=s, kosul=kosul, kar=v["kar"], roi=v["roi"],
            ece=v["ece"], esik=v["esik"], dkar=v["dkar"], yuzde=v["dkar_yuzde"]))
    yaz(f"  ({time.time()-t:.0f}s)")
abl_df = rb.tablo_ablation(tum_abl)
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "ablation_profit.csv"))

===== PART 1 — PROFIT-CHAIN ABLATION =====


telco / K0: profit=543647 ROI=1.32 ECE=0.009 t*=0.05 | Δvs K0=+0 (+0.0%)
telco / K1: profit=-1075056 ROI=-9.62 ECE=0.009 t*=0.50 | Δvs K0=-1618703 (-297.7%)
telco / K2: profit=541433 ROI=1.36 ECE=0.131 t*=0.11 | Δvs K0=-2215 (-0.4%)
telco / K3: profit=525916 ROI=1.25 ECE=0.013 t*=0.04 | Δvs K0=-17732 (-3.3%)
telco / K4: profit=542918 ROI=1.34 ECE=0.083 t*=0.06 | Δvs K0=-729 (-0.1%)
  (68s)


cell2cell / K0: profit=2537396 ROI=0.71 ECE=0.005 t*=0.05 | Δvs K0=+0 (+0.0%)
cell2cell / K1: profit=-17477175 ROI=-75.84 ECE=0.005 t*=0.50 | Δvs K0=-20014571 (-788.8%)
cell2cell / K2: profit=2537676 ROI=0.71 ECE=0.177 t*=0.11 | Δvs K0=+280 (+0.0%)
cell2cell / K3: profit=2536017 ROI=0.70 ECE=0.008 t*=0.01 | Δvs K0=-1379 (-0.1%)
cell2cell / K4: profit=2539123 ROI=0.71 ECE=0.030 t*=0.07 | Δvs K0=+1727 (+0.1%)
  (408s)
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/tables/ablation_profit.csv


## Part 2 — Repeated runs + 95% CI (5 sets)

In [3]:
yaz(S.MSG["bolum"].format(ad="PART 2 — REPEATED RUNS + CI"))
tum_ci = {}
for s in cfg.DATASETS:
    t = time.time()
    tum_ci[s] = rb.robustness_set(s, veriler[s])
    for metrik, (m, std, lo, hi) in tum_ci[s].items():
        yaz(S.MSG7["ci"].format(set=s, metrik=metrik, ort=m, std=std, lo=lo, hi=hi))
    yaz(f"  ({time.time()-t:.0f}s)")
ci_df = rb.tablo_ci(tum_ci)
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "robustness_ci.csv"))

===== PART 2 — REPEATED RUNS + CI =====


telco PR-AUC: 0.6639 ± 0.0014  [95% CI 0.6621, 0.6657]
telco ROC-AUC: 0.8473 ± 0.0004  [95% CI 0.8467, 0.8478]
telco recall: 0.9832 ± 0.0027  [95% CI 0.9799, 0.9865]
telco precision: 0.3462 ± 0.0077  [95% CI 0.3367, 0.3557]
telco F1: 0.5120 ± 0.0081  [95% CI 0.5020, 0.5221]
telco EMP: 0.0497 ± 0.0005  [95% CI 0.0491, 0.0503]
  (1s)


cell2cell PR-AUC: 0.4659 ± 0.0010  [95% CI 0.4647, 0.4672]
cell2cell ROC-AUC: 0.6852 ± 0.0007  [95% CI 0.6843, 0.6861]
cell2cell recall: 0.9998 ± 0.0001  [95% CI 0.9997, 1.0000]
cell2cell precision: 0.2885 ± 0.0002  [95% CI 0.2883, 0.2887]
cell2cell F1: 0.4478 ± 0.0002  [95% CI 0.4475, 0.4481]
cell2cell EMP: 0.0352 ± 0.0000  [95% CI 0.0352, 0.0352]
  (7s)


ecommerce PR-AUC: 0.9035 ± 0.0075  [95% CI 0.8942, 0.9128]
ecommerce ROC-AUC: 0.9656 ± 0.0045  [95% CI 0.9600, 0.9712]
ecommerce recall: 0.9080 ± 0.0076  [95% CI 0.8986, 0.9174]
ecommerce precision: 0.7127 ± 0.0112  [95% CI 0.6989, 0.7266]
ecommerce F1: 0.7986 ± 0.0089  [95% CI 0.7875, 0.8096]
ecommerce EMP: 0.0153 ± 0.0015  [95% CI 0.0134, 0.0171]
  (76s)


iranian PR-AUC: 0.9549 ± 0.0027  [95% CI 0.9515, 0.9582]
iranian ROC-AUC: 0.9889 ± 0.0012  [95% CI 0.9874, 0.9904]
iranian recall: 0.9657 ± 0.0065  [95% CI 0.9575, 0.9738]
iranian precision: 0.7630 ± 0.0303  [95% CI 0.7254, 0.8007]
iranian F1: 0.8521 ± 0.0166  [95% CI 0.8315, 0.8727]
iranian EMP: 0.0004 ± 0.0003  [95% CI 0.0001, 0.0008]
  (88s)


bank PR-AUC: 0.7031 ± 0.0019  [95% CI 0.7007, 0.7054]
bank ROC-AUC: 0.8659 ± 0.0007  [95% CI 0.8651, 0.8668]
bank recall: 0.9680 ± 0.0049  [95% CI 0.9619, 0.9741]
bank precision: 0.2711 ± 0.0105  [95% CI 0.2580, 0.2842]
bank F1: 0.4234 ± 0.0122  [95% CI 0.4083, 0.4386]
bank EMP: 0.0317 ± 0.0005  [95% CI 0.0311, 0.0323]
  (22s)
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/tables/robustness_ci.csv


## Part 3 — Significance (Wilcoxon, paired scores)

In [4]:
yaz(S.MSG["bolum"].format(ad="PART 3 — SIGNIFICANCE"))
sig_df, pmat, modeller, n = rb.anlamlilik(veriler)
yaz(S.MSG7["yontem"].format(n=n))
yaz(sig_df.to_string(index=False))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "significance_tests.csv"))

===== PART 3 — SIGNIFICANCE =====


Significance method: Wilcoxon signed-rank over paired scores (dataset×seed OOF PR-AUC, n=25).
                                     Comparison     Test  Statistic  p-value               Result
                        lgbm vs logreg (PR-AUC) Wilcoxon        0.0  0.00000 significant (p<0.05)
                       lgbm vs xgboost (PR-AUC) Wilcoxon      117.0  0.23036       noise (p≥0.05)
                     logreg vs xgboost (PR-AUC) Wilcoxon        0.0  0.00000 significant (p<0.05)
Profit threshold (K0) vs accuracy (K1) — profit Wilcoxon        0.0  0.00195 significant (p<0.05)
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/tables/significance_tests.csv


## Figures

In [5]:
yaz(S.MSG["bolum"].format(ad="FIGURES"))
y1 = rb.figur_ablation(tum_abl)
y2 = rb.figur_ci(tum_ci)
y3 = rb.figur_significance(pmat, modeller)
for y in (y1, y2, y3):
    yaz(S.MSG["kayit"].format(yol=y))

===== FIGURES =====
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_robust/ablation_profit_waterfall.png
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_robust/robustness_ci_forest.png
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_robust/significance_heatmap.png


## Summary — decision left to the user

In [6]:
yaz(S.MSG["bolum"].format(ad="SUMMARY — ROBUSTNESS"))
yaz("ABLATION (profit loss % vs K0):")
for s in rb.ABLATION_SETLERI:
    d = tum_abl[s]
    sirali = sorted([(k, d[k]["dkar_yuzde"]) for k in d if k != "K0"], key=lambda x: x[1])
    yaz(f"  {s}: " + " | ".join(f"{k}={dy:+.0f}%" for k, dy in sirali))
yaz("\nCI width (PR-AUC, max std):")
maxstd = max(tum_ci[s]["PR-AUC"][1] for s in cfg.DATASETS)
yaz(f"  widest PR-AUC std = {maxstd:.4f} -> {'very tight, results stable' if maxstd < 0.02 else 'caution'}")
yaz("\nSIGNIFICANCE:")
for _, r in sig_df.iterrows():
    yaz(f"  {r[S.KOLON7['kiyas']]}: p={r[S.KOLON7['p']]} -> {r[S.KOLON7['sonuc']]}")
yaz("")
yaz(S.MSG7["bitti"])

_log = cfg.LOGS / "adim7_ozet.log"
_log.write_text("\n".join(CIKTI) + "\n", encoding="utf-8")
print(S.MSG["kayit"].format(yol=_log))

===== SUMMARY — ROBUSTNESS =====
ABLATION (profit loss % vs K0):
  telco: K1=-298% | K3=-3% | K2=-0% | K4=-0%
  cell2cell: K1=-789% | K3=-0% | K2=+0% | K4=+0%

CI width (PR-AUC, max std):
  widest PR-AUC std = 0.0075 -> very tight, results stable

SIGNIFICANCE:
  lgbm vs logreg (PR-AUC): p=0.0 -> significant (p<0.05)
  lgbm vs xgboost (PR-AUC): p=0.23036 -> noise (p≥0.05)
  logreg vs xgboost (PR-AUC): p=0.0 -> significant (p<0.05)
  Profit threshold (K0) vs accuracy (K1) — profit: p=0.00195 -> significant (p<0.05)

STEP 7 (robustness) complete. Experimental phase done; interpretation left to the user, write-up next.
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/logs/adim7_ozet.log
